# AeroPulse Propagation — 01: Data Contract & Quality Gates

Import the upstream event-aware table, prove it satisfies a machine-readable
contract, and refuse to hand a bad dataset to the rest of the pipeline.

## Objective

Improvement plan sections 4 and 52 (Notebook 01). The previous version of this
notebook checked that the horizon target columns existed and wrote a
fingerprint. That is necessary but nowhere near sufficient: a dataset can have
every required column and still be unusable.

This notebook adds:

- a **machine-readable contract** (`pt.SCHEMA`, schema version 2.0.0) covering
  required columns, units, and physical ranges — training fails fast on a violation
- **data quality gates**: duplicates, ordering, hourly continuity, negative and
  impossible values, coordinate validation, long missing-data runs, target availability
- a **station-level coverage report**, because a dataset that looks 98% complete
  in aggregate can still contain stations with three weeks of data
- `data_quality_report.json` and a dataset fingerprint

One thing to state plainly up front: **wind is km/h, not m/s.** Open-Meteo is
queried without `wind_speed_unit`, so its default (km/h) applies. The contract
records that unit so the transport notebook cannot silently reintroduce the
3.6x error the previous implementation carried.

In [1]:
# ============================================================================
# SETUP — everything reusable lives in propagation_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import propagation_toolkit as pt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

cfg = pt.load_config()
HORIZONS = cfg.horizons
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast
horizons: [1, 3, 6, 12, 24, 48] | schema 2.0.0
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


## 1. Load the upstream event-aware table

In [2]:
t0 = time.time()
if not cfg.event_aware_file.exists():
    raise FileNotFoundError(
        f"{cfg.event_aware_file} not found. Propagation reuses the pm25 Phase-2 "
        "targets — run the pm25 estimator pipeline first."
    )

df = pd.read_parquet(cfg.event_aware_file)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

print(f"loaded in {time.time() - t0:.1f}s")
print("rows      ", f"{len(df):,}")
print("columns   ", df.shape[1])
print("stations  ", df["location_id"].nunique())
print("window    ", df["timestamp_utc"].min(), "->", df["timestamp_utc"].max())
print("memory    ", f"{df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

loaded in 0.3s
rows       1,705,252
columns    208
stations   149
window     2025-02-18 20:30:00+00:00 -> 2026-09-09 07:30:00+00:00
memory     1.58 GB


## 2. Contract validation

`pt.validate_contract` raises on a missing required column or a missing horizon
target. There is no "warn and continue" path — a contract that can be ignored
is documentation, not a contract.

In [3]:
contract = pt.validate_contract(df, cfg)
print(json.dumps(contract, indent=2))

print("\nunits declared by the contract:")
for column, unit in pt.SCHEMA["units"].items():
    print(f"  {column:<24} {unit}")

{
  "schema_version": "2.0.0",
  "missing_required": [],
  "missing_targets": [],
  "absent_recommended": [],
  "rows": 1705252,
  "columns": 208,
  "status": "PASS"
}

units declared by the contract:
  pm25                     ug/m3
  wind_speed_10m           km/h
  wind_u                   km/h
  wind_v                   km/h
  boundary_layer_height    m
  temperature_2m           degC
  surface_pressure         hPa


### Contract negative test

A contract nobody has watched fail is not known to work. Drop a required
column from a copy and confirm the validator refuses it.

In [4]:
broken = df.drop(columns=["wind_u"]).head(1000)
try:
    pt.validate_contract(broken, cfg)
    print("FAIL — the validator accepted a dataset with no wind_u")
except ValueError as exc:
    print("PASS — contract rejected the broken dataset:")
    print(" ", exc)

PASS — contract rejected the broken dataset:
  Data contract violated. missing_required=['wind_u'] missing_targets=[]


## 3. Data quality gates

In [5]:
report, station_coverage = pt.quality_report(df, cfg)
core = {k: v for k, v in report.items() if k not in {"out_of_range", "target_availability_pct", "failures"}}
print(json.dumps(core, indent=2))

print("\nout-of-range counts:")
print(json.dumps(report["out_of_range"], indent=2) if report["out_of_range"] else "  none")

print("\ntarget availability (% of rows with a usable label):")
for name, pct in report["target_availability_pct"].items():
    print(f"  {name:<28} {pct:6.2f}%")

print("\nQUALITY STATUS:", report["quality_status"])
if report["failures"]:
    for failure in report["failures"]:
        print("  FAIL:", failure)

{
  "generated_utc": "2026-09-10T12:13:56.728479+00:00",
  "rows": 1705252,
  "stations": 149,
  "window_start": "2025-02-18 20:30:00+00:00",
  "window_end": "2026-09-09 07:30:00+00:00",
  "missing_pm25_pct": 0.0,
  "duplicate_rows": 0,
  "timestamp_gaps": 56609,
  "gap_hours_max": 7320.0,
  "long_gaps_over_24h": 1984,
  "invalid_pm25_rows": 0,
  "stations_off_grid": 0,
  "station_completeness_p10": 75.112,
  "station_completeness_median": 86.41,
  "stations_under_50pct_complete": 1,
  "quality_status": "PASS"
}

out-of-range counts:
  none

target availability (% of rows with a usable label):
  target_pm25_t_plus_1h         96.67%
  target_pm25_t_plus_3h         94.52%
  target_pm25_t_plus_6h         92.91%
  target_pm25_t_plus_12h        91.06%
  target_pm25_t_plus_24h        89.96%
  target_pm25_t_plus_48h        88.37%

QUALITY STATUS: PASS


### Station coverage

Aggregate completeness hides the stations that will produce unreliable
features. `completeness_pct` is observed hours over the station's own
first-to-last span, so a station that joined late is not punished for the
months before it existed.

In [6]:
print("stations:", len(station_coverage))
print(station_coverage["completeness_pct"].describe().round(2).to_string())

worst = station_coverage.nsmallest(10, "completeness_pct")[
    ["location_id", "rows", "completeness_pct", "first", "last"]
]
print("\nten least complete stations:")
print(worst.to_string(index=False))

thin = station_coverage[station_coverage["rows"] < 720]  # under 30 days
print(f"\nstations with under 30 days of data: {len(thin)}")

stations: 149
count    149.00
mean      84.14
std        7.95
min       44.64
25%       82.05
50%       86.41
75%       89.33
max       96.61

ten least complete stations:
 location_id  rows  completeness_pct                     first                      last
        8558  6079             44.64 2025-02-18 20:30:00+00:00 2026-09-09 06:30:00+00:00
     3409439  7099             52.19 2025-02-18 20:30:00+00:00 2026-09-08 13:30:00+00:00
     3409395  7930             58.30 2025-02-18 20:30:00+00:00 2026-09-08 13:30:00+00:00
     3410005  8354             61.42 2025-02-18 20:30:00+00:00 2026-09-08 13:30:00+00:00
      270703  8468             62.26 2025-02-18 20:30:00+00:00 2026-09-08 13:30:00+00:00
     3409513  8609             63.29 2025-02-18 20:30:00+00:00 2026-09-08 13:30:00+00:00
     3409441  8943             65.75 2025-02-18 20:30:00+00:00 2026-09-08 13:30:00+00:00
     3409474  9046             66.50 2025-02-18 20:30:00+00:00 2026-09-08 13:30:00+00:00
     3409493  9162         

## 4. Gap structure

Hourly continuity matters for lag and rolling features: a 40-hour hole means
`pm25_lag_24h` is silently filled from the wrong side of the gap.

In [7]:
ordered = df.sort_values(["location_id", "timestamp_utc"])
gap_h = ordered.groupby("location_id", observed=True)["timestamp_utc"].diff().dt.total_seconds() / 3600.0
gaps = gap_h[gap_h > 1.0]

print(f"gap events        {len(gaps):,}")
print(f"largest gap       {gaps.max():.0f} h" if len(gaps) else "no gaps")
print(f"gaps over 6 h     {(gaps > 6).sum():,}")
print(f"gaps over 24 h    {(gaps > 24).sum():,}")
print(f"hours lost to gaps{(gaps - 1).sum():,.0f}")

buckets = pd.cut(gaps, [1, 2, 6, 24, 72, np.inf], labels=["1-2h", "2-6h", "6-24h", "1-3d", ">3d"])
print("\ngap size distribution:")
print(buckets.value_counts().sort_index().to_string())

gap events        56,609
largest gap       7320 h
gaps over 6 h     10,318
gaps over 24 h    1,984
hours lost to gaps321,494

gap size distribution:
timestamp_utc
1-2h     28524
2-6h     17767
6-24h     8334
1-3d      1590
>3d        394


## 5. Feature availability metadata

Plan section 5: every feature needs `observation_time`, `available_time`, and
must satisfy `available_time <= forecast_time`. This dataset is assembled from
sources with different latencies, so we record the assumption explicitly
rather than leaving it implied by the column name.

The critical entry is **weather**: this table holds *observed* weather at time
t only. It contains no observation valid after t, which is what makes the
current feature set leakage-free. Notebook 03 is where forecast-vintage
handling becomes a live concern, because advection needs wind *after* t.

In [8]:
AVAILABILITY = {
    "pm25":       {"kind": "observation", "latency_minutes": 60,
                   "note": "CPCB/OpenAQ hourly, available at the end of the hour"},
    "pm25_lag_*": {"kind": "observation", "latency_minutes": 60, "note": "strictly historical"},
    "pm25_roll_*":{"kind": "derived",     "latency_minutes": 60, "note": "past-only windows, verified in 02"},
    "weather_*":  {"kind": "observation", "latency_minutes": 60,
                   "note": "ERA5/Open-Meteo reanalysis valid at t; NO post-t values present"},
    "wind_u/v":   {"kind": "observation", "latency_minutes": 60, "units": "km/h"},
    "fire_*":     {"kind": "observation", "latency_minutes": 180,
                   "note": "VIIRS/MODIS overpass; 24 h aggregation windows end at t"},
    "cams_*":     {"kind": "analysis",    "latency_minutes": 300,
                   "note": "CAMS analysis; satellite_age_hours records staleness"},
    "regional_*/upwind_*": {"kind": "derived", "latency_minutes": 60,
                   "note": "neighbour field at t, leave-one-out"},
    "target_*":   {"kind": "label",       "latency_minutes": None,
                   "note": "future observation; never a feature"},
}
for name, meta in AVAILABILITY.items():
    print(f"{name:<22} {meta['kind']:<12} latency={str(meta['latency_minutes']):>5} min")

future_valid_columns = [c for c in df.columns if c.startswith("target_")]
print(f"\ncolumns valid after t (labels only): {len(future_valid_columns)}")
print("weather columns valid after t:", 0, "— nothing in this table is a forecast")

pm25                   observation  latency=   60 min
pm25_lag_*             observation  latency=   60 min
pm25_roll_*            derived      latency=   60 min
weather_*              observation  latency=   60 min
wind_u/v               observation  latency=   60 min
fire_*                 observation  latency=  180 min
cams_*                 analysis     latency=  300 min
regional_*/upwind_*    derived      latency=   60 min
target_*               label        latency= None min

columns valid after t (labels only): 14
weather columns valid after t: 0 — nothing in this table is a forecast


## 6. Persist the base dataset

Downcast to float32 (halves the memory for the rest of the pipeline with no
measurable effect on tree models) and write the base table plus manifest.

In [9]:
float_cols = df.select_dtypes(include=["float64"]).columns
df[float_cols] = df[float_cols].astype("float32")
print(f"downcast {len(float_cols)} float64 columns -> float32")
print("memory now", f"{df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

out = cfg.base_dir / "station_pm25_series.parquet"
df.to_parquet(out, index=False)
station_coverage.to_csv(cfg.base_dir / "station_coverage.csv", index=False)

fingerprint = pt.dataset_fingerprint(df[["timestamp_utc", "location_id", "pm25"]])
report["fingerprint"] = fingerprint
(cfg.base_dir / "data_quality_report.json").write_text(json.dumps(report, indent=2, default=str))

manifest = {
    "schema_version": cfg.schema_version,
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "source": "pm25_phase2_event_aware",
    "source_path": str(cfg.event_aware_file),
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "stations": int(df["location_id"].nunique()),
    "window": [str(df["timestamp_utc"].min()), str(df["timestamp_utc"].max())],
    "horizons": HORIZONS,
    "contract": contract,
    "quality_status": report["quality_status"],
    "feature_availability": AVAILABILITY,
    "fingerprint": fingerprint,
    "git_commit": pt.git_commit(),
}
(cfg.base_dir / "dataset_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))

print("\n01 COMPLETE")
print("  ", out)
print("   fingerprint", fingerprint[:16], "...")
print("Next: 02_propagation_temporal_spatial_features.ipynb")

downcast 0 float64 columns -> float32
memory now 1.58 GB



01 COMPLETE
   /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/data/propagation/processed/base/station_pm25_series.parquet
   fingerprint a9520e88c0ed9b98 ...
Next: 02_propagation_temporal_spatial_features.ipynb


## Findings

**Contract: PASS.** All 13 required columns and all six horizon targets are
present across 1,705,252 rows / 149 stations, 2025-02-18 to 2026-09-09. No
column violates its declared physical range. The negative test confirms the
validator actually rejects a dataset missing `wind_u` rather than warning
about it.

**Quality gates: PASS.** No duplicate `(location_id, timestamp_utc)` pairs, no
negative or impossible PM2.5, no station outside the India bounding box, zero
missing PM2.5 on the rows that exist. Median station completeness 86.4%, p10
station 75%, and no station with under 30 days of data.

**The real defect is hourly continuity, not validity:**

| gap size | events |
|---|---|
| 1-2 h | 28,524 |
| 2-6 h | 17,767 |
| 6-24 h | 8,334 |
| 1-3 d | 1,590 |
| > 3 d | 394 |

56,609 gap events, 321,494 hours absent from the nominal hourly grid, largest
single gap 7,320 h (305 days).

Checked rather than assumed: the upstream lag columns are **time-aligned, not
positional**. `pm25_lag_24h` matches an exact timestamp-minus-24 h lookup on
100% of non-null rows (a positional `shift(24)` would only agree on 56%). So
the gaps do not corrupt the lags — they *empty* them. The cost lands as
missingness that rises with lag depth:

| feature | null % |
|---|---|
| `pm25_lag_1h` | 3.3 |
| `pm25_lag_24h` | 10.0 |
| `pm25_lag_168h` | 14.8 |
| `pm25_roll_mean_24h` | 1.1 |
| `regional_pm25_mean_50km` | 34.0 |
| `upwind_pm25_mean` | 36.3 |

That directly sets an expectation for notebook 04: the 168 h seasonal
baseline is unavailable for roughly one row in seven, so the baseline
ensemble must renormalise its weights rather than treating a missing member
as zero. It also means the 36% null rate on the upwind/regional field — a
neighbour-density artefact, not a gap artefact — is the single largest
missingness in the feature set, and the imputation strategy for it is a
modelling decision, not a detail.

**Target availability** falls from 96.7% at 1 h to 88.4% at 48 h. The 48 h
horizon trains on ~8% fewer rows than 1 h, and those rows are not missing at
random: they cluster at the end of each station's series and around the gaps.

**Feature availability.** No column in this table is valid after time t; the
only post-t values are the 14 `target_*` labels. Weather is observed
reanalysis valid at t, not a forecast. That is what makes the current feature
set leakage-free — and exactly why notebook 03 cannot simply read the wind at
t+h when integrating advection.